# R2·R3 집단 라우터 추가 — G 평가

**런타임: CPU, Python 3.12.** 학습·모델 추론·LLM API 호출이 없습니다.
`12a_r2_group.ipynb`와 `12b_r3_group.ipynb`의 학습·점수 저장을 먼저 끝냅니다.

기존 비교군은 `run001/gain-lock`에 고정된 설정·보정·λ를 그대로 재사용합니다.
R2·R3만 dev로 같은 5분할 isotonic 보정을 하고 비용 수준별 λ를 고정합니다.
새로운 설정 탐색은 하지 않습니다. 주 결과는 τ=.8, seed 42입니다.

**G = A − H(C)**: A는 첫 시도 해결 확률, C는 첫 호출 비용,
H(C)는 같은 비용 이하의 고정 단계 또는 무작위 혼합이 얻는 최선의 해결 확률입니다.
기존 G 구현을 그대로 사용합니다. 에스컬레이션을 포함하는 효용 U와는 다른 지표입니다.

비용 1:3:5가 주 결과이며 τ=.8에서는 집단 평균 비용과 길이 반영 비용도 계산합니다.
τ 곡선·비용 가중치 λ별 비교·원천별/전체 표·쌍별 차이·95% 구간을 출력합니다.
paired bootstrap 10,000회이며 기존 비교군의 점수와 구간 재현을 검사합니다.

기존 test를 재사용하는 후속 비교입니다. 결과를 보고 학습 설정이나 λ를 다시 맞추지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
REPO = Path('/content/router-final-r23-source')
SOURCE_ZIP = RUN / 'code/router-final-source-r23.zip'
EXPECTED_SHA = 'd5dac9c40bac97ce05b7ca7e4cd07fcc4ffe9c703b2874c376ff9753dedee62e'
if not SOURCE_ZIP.is_file():
    from google.colab import files
    print('배포 묶음 안의 router-final-source-r23.zip을 선택하세요.')
    uploaded = files.upload()
    SOURCE_ZIP = Path('/content/router-final-source-r23.zip')
    SOURCE_ZIP.write_bytes(uploaded['router-final-source-r23.zip'])
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(SOURCE_ZIP) == EXPECTED_SHA, '노트북과 소스 ZIP의 버전이 다릅니다.'
with zipfile.ZipFile(SOURCE_ZIP) as bundle:
    for name in bundle.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    bundle.extractall(REPO)
for name, digest in json.loads((REPO / 'source-bundle-manifest.json').read_text()).items():
    assert sha(REPO / name) == digest, name
assert sys.version_info[:2] in {(3, 11), (3, 12)}, 'Python 3.12 런타임을 선택하세요.'


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'scikit-learn==1.6.1', 'numpy==2.1.2', 'pandas==2.3.3', 'pyarrow==21.0.0'], check=True)


계산은 별도 Python 프로세스에서 실행하므로 설치 후 세션을 다시 시작할 필요가 없습니다. 아래 경로가 기존 실험의 Drive 폴더인지 확인하세요.


In [ ]:
from pathlib import Path
import os, subprocess, sys
SCRIPT = REPO / 'ml/notebooks/router_final/r23_extension.py'
ENV = dict(os.environ, PYTHONPATH=str(REPO / 'ml/src'), PYTHONUNBUFFERED='1',
           TOKENIZERS_PARALLELISM='false')
def run_logged(command, *args):
    cmd = [sys.executable, str(SCRIPT), command, '--run-dir', str(RUN), '--repo', str(REPO),
           *map(str, args)]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=ENV)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait():
        raise RuntimeError(f'{command} 실패 (exit {proc.returncode}). 위 오류를 확인하세요.')
run_logged('preflight')


## 1. R2·R3 보정과 λ 고정 — dev만 사용

기존 라우터는 다시 선택하거나 보정하지 않습니다. 아래 요약은 test 평가 전에 기록됩니다.


In [ ]:
LOCK = RUN / 'r23-extension/gain-lock'
if not LOCK.exists():
    run_logged('lock')
print((LOCK / 'gain-lock-summary.md').read_text())


## 2. dev 리허설

경로·평가 코드를 검사합니다. dev로 보정한 값을 dev에서 평가하므로 성능 결과로 사용하지 않습니다.


In [ ]:
import tempfile
REHEARSAL = Path(tempfile.mkdtemp(prefix='r23-rehearsal-', dir='/content')) / 'results'
run_logged('rehearse', '--output-dir', REHEARSAL)
print((REHEARSAL / 'gain-summary.md').read_text()[:3000])


## 3. 고정한 정책으로 test 평가

원본 `gain-test`를 유지하고 `r23-extension/gain-test`에 새 결과를 씁니다. 기존 라우터의 원천별 A·C·H·G·선택 비율·95% 구간이 이전과 다르면 검사에서 실패합니다.


In [ ]:
OUT = RUN / 'r23-extension/gain-test'
assert not OUT.exists(), '이미 출력이 있습니다. 완료 결과를 읽거나, 실패 로그를 확인한 뒤 미완료 폴더를 별도 보관하세요.'
run_logged('test')


In [ ]:
import pandas as pd, json
print((OUT / 'gain-summary.md').read_text())
display(pd.read_csv(OUT / 'r0-r3-comparison.csv'))
print('기존 결과 재현:', json.loads((OUT / 'manifest.json').read_text())['baseline_point_and_interval_reproduced'])


## 4. 결과 다운로드

표와 manifest, 요약만 묶습니다. 모델 가중치나 질문 원문은 포함하지 않습니다.


In [ ]:
import zipfile
from google.colab import files
RESULT_ZIP = Path('/content/r23-group-gain-results.zip')
with zipfile.ZipFile(RESULT_ZIP, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUT.iterdir()):
        if path.suffix in {'.csv', '.md', '.json', '.parquet'}:
            bundle.write(path, path.name)
files.download(str(RESULT_ZIP))
